# Templisafe - library quickstart

## Minimal working example

### Template

In [ ]:
from templisafe.util.util import ContentType
from templisafe.settings.source_settings import SourceSettings

sql_template_content: str = """SELECT 
{%- for col in select_columns %}
  {{ col }}{% if not loop.last %},{% endif %}
{%- endfor %}
FROM users u
  JOIN metrics m ON u.{{ user_join_key }} = m.{{ metric_join_key }}
WHERE TRUE 
  AND u.code IN ({{ user_codes | join(', ') }})
  AND u.age BETWEEN {{ user_age_lower }} AND {{ user_age_upper }}
  AND u.status = '{{ user_status }}'
  AND m.code = '{{ metric_code }}'
  AND m.is_updated IS {{ metric_updated_flag }}
  AND m.threshold > {{ metric_threshold }}
"""

template_inline_source_settings: SourceSettings = SourceSettings.create(
    kind="inline", 
    content=sql_template_content, 
    content_type=ContentType.TEXT
)
template_inline_source_settings

### Schema

In [ ]:
schema_content: str = """
schema:
  select_columns:
    type: list
    default:
      - u.id
      - u.name
      - u.code
      - u.age
      - u.status
      - m.is_updated
      - m.threshold
    metadata:
      description: Target select list
      title: SELECT LIST

  user_join_key: 
    type: str
    metadata:
      description: Join key for table user

  metric_join_key: str
    
  user_codes: 
    type: list
    default:
      - 10
      - 11
      - 12
    
  user_age_lower:
    type: int
    default: 0
    constraints:
      ge: 0

  user_age_upper:
    type: int
    default: 1000000
    constraints:
      ge: 0

  user_status: str

  metric_code:
    type: str
    default: "9999910"
    constraints:
      max_length: 12

  metric_updated_flag:
    type: bool
    default: true

  metric_threshold: float
"""

schema_inline_source_settings: SourceSettings = SourceSettings.create(
    kind="inline", 
    content_type=ContentType.YAML, 
    content=schema_content
 )
schema_inline_source_settings

### Variants

In [ ]:
variants_content: str = """
variants:
  select_columns:
    - u.id
    - u.name
    - u.age
    - m.value

  user_join_key: id
  metric_join_key: user_id
  # user_codes -> defaulted
    
  user_age_lower:  18
  user_age_upper: 70
  user_status: ACTIVE

  # metric_code -> defaulted
  metric_updated_flag: true
  metric_threshold: 54.12
"""
variants_inline_source_settings: SourceSettings = SourceSettings.create(
    kind="inline", 
    content_type=ContentType.YAML, 
    content=variants_content
)
variants_inline_source_settings

### Compilation

#### Create the `Templater`

In [ ]:
from templisafe.templater import Templater
from templisafe.templater_factory import TemplaterFactory

factory: TemplaterFactory = TemplaterFactory()

templater: Templater = factory.create()
templater

#### Compile

In [ ]:
from templisafe.templater import Compilation

compilation: Compilation = templater.compile(
    template_source=template_inline_source_settings,
    schema_source=schema_inline_source_settings
)

compilation.outcome

In [ ]:
compilation

The schema generated is nothing but a **pydantic model**. 

In [ ]:
compilation.compiled.schema.model_cls

In [ ]:
compilation.compiled.schema.model_cls.model_fields

#### Compile without a `Schema`

When no schema is provided, all variables are parsed as `object` with default to `None`.

In [ ]:
compilation_empty_schema: Compilation = templater.compile(template_source=template_inline_source_settings)      # No schema provided
compilation_empty_schema.outcome

In [ ]:
compilation_empty_schema.message

In [ ]:
from templisafe.template.template_model import CompilationSpec, Schema

compiled_empty_schema: CompilationSpec = compilation_empty_schema.compiled
empty_schema: Schema = compiled_empty_schema.schema
empty_schema.model_cls.model_fields

### Rendering

In [ ]:
from templisafe.settings.source_settings import InlineSourceSettings

assert isinstance(variants_inline_source_settings, InlineSourceSettings)
print(variants_inline_source_settings.content)

In [ ]:
from templisafe.templater import Rendering

rendering: Rendering = templater.render(
    compiled=compilation.compiled,
    variants_sources=variants_inline_source_settings
)

rendering.outcome

If the variant has no name associated, a default one is generated.

In [ ]:
from templisafe.template.template_model import RenderingSpec

rendered: RenderingSpec = rendering.rendered 
rendered.names

In [ ]:
for r in rendered.parameterizations: 
    print(r.rendered_str)

In [ ]:
rendered.parameterizations[0].variant.names

In [ ]:
rendered.parameterizations[0].variant.bindings

### Build

#### Configuration files

In [ ]:
TEMPLATE_PATH: str = "./template.sql.j2"
with open(TEMPLATE_PATH) as f:
    print(f.read())

In [ ]:
SCHEMA_PATH: str = "./schema.yaml"
with open(SCHEMA_PATH) as f:
    print(f.read())


In [ ]:
VARIANTS_PATH_1: str = "./variants1.yaml"
with open(VARIANTS_PATH_1) as f:
    print(f.read())

In [ ]:
VARIANTS_PATH_2: str = "./variants2.yaml"
with open(VARIANTS_PATH_2) as f:
    print(f.read())

#### Sources

Use a `LocalSource` to load configurations from a file:

In [ ]:
template_local_source_settings: SourceSettings = SourceSettings.create(
    kind="local", path=TEMPLATE_PATH
)

schema_local_source_settings: SourceSettings = SourceSettings.create(
    kind="local", path=SCHEMA_PATH
)

variants1_local_source_settings: SourceSettings = SourceSettings.create(
    kind="local", path=VARIANTS_PATH_1
)

variants2_local_source_settings: SourceSettings = SourceSettings.create(
    kind="local", path=VARIANTS_PATH_2
)

template_local_source_settings, schema_local_source_settings, variants1_local_source_settings, variants2_local_source_settings

#### Build (compilation + rendering)

Use **build** to **compile** and **render** in one step:

In [ ]:
from templisafe.templater import Build

build: Build = templater.build(
    template_source=template_local_source_settings,
    schema_source=schema_local_source_settings,
    variants_sources=[variants1_local_source_settings, variants2_local_source_settings]
)

build.outcome

In [ ]:
compilation: Compilation = build.compilation
compilation.outcome, compilation.message

In [ ]:
rendering: Rendering = build.rendering
rendering.outcome, rendering.message

In [ ]:
rendering.rendered.names

In [ ]:
from templisafe.template.template_model import Parameterization, Variant

parameterizations: list[Parameterization] = rendering.rendered.parameterizations
for par in parameterizations:
    variant: Variant = par.variant
    print("-" * 50)
    print(f"Variant '{variant.name}':")
    print("-" * 50)
    for b in variant.bindings:
        print(b)

In [ ]:
for variant_name, variant in rendering.rendered.mapping.items(): 
    print("-" * 50)
    print(f"Variant '{variant_name}':")
    print("-" * 50)
    print(variant.rendered_str)

## Query diagnostics

### Compilation

#### Undeclared variables

Template

In [ ]:
from templisafe.util.util import ContentType
from templisafe.settings.source_settings import InlineSourceSettings

sql_template_content: str = """SELECT 
  {{ col1 }}, {{ col2 }}
FROM users u
WHERE TRUE
  AND u.status = '{{ user_status }}'
  AND u.age > {{ user_age_lower }}
  AND m.code = '{{ undeclared }}'
"""

template_inline_source_settings: InlineSourceSettings = InlineSourceSettings(content=sql_template_content, content_type=ContentType.TEXT)
template_inline_source_settings

Schema

In [ ]:
from templisafe.util.util import ContentType
from templisafe.settings.source_settings import InlineSourceSettings

# Parameter 'undeclared' not declared in the schema
schema_content: str = """schema:
  col1: str
  col2: str
  user_status: str
  user_age_lower: int 
"""

schema_inline_source_settings: SourceSettings = SourceSettings.create(
    kind='inline',
    content=schema_content,
    content_type=ContentType.YAML
)

schema_inline_source_settings

In [ ]:
from templisafe.templater import Templater
from templisafe.templater_factory import TemplaterFactory

factory: TemplaterFactory = TemplaterFactory()
templater: Templater = factory.create(diagnostic_policy="ignore")        # Use ignore policy to avoid raising errors

compilation: Compilation = templater.compile(
    template_source=template_inline_source_settings,
    schema_source=schema_inline_source_settings
)

compilation.outcome

In [ ]:
compilation.message, compilation.diagnostics

#### Unused variables

Template

In [ ]:
from templisafe.util.util import ContentType

sql_template_content: str = """SELECT 
  {{ col1 }}, {{ col2 }}
FROM users u
WHERE TRUE
  AND u.status = '{{ user_status }}'
  AND u.age > {{ user_age_lower }}
"""

template_inline_source_settings: SourceSettings = SourceSettings.create(
    kind='inline',
    content=sql_template_content, 
    content_type=ContentType.TEXT
    )
template_inline_source_settings

Schema

In [ ]:
from templisafe.util.util import ContentType

# Parameter 'undeclared' not declared in the schema
schema_content: str = """schema:
  col1: str
  col2: str
  user_status: str
  user_age_lower: int 
  unused: any
"""

schema_inline_source_settings: SourceSettings = SourceSettings.create(
    kind='inline',
    content=schema_content,
    content_type=ContentType.YAML
)

schema_inline_source_settings

In [ ]:
from templisafe.templater import Templater
from templisafe.templater_factory import TemplaterFactory

factory: TemplaterFactory = TemplaterFactory()
templater: Templater = factory.create(diagnostic_policy="log")       # Use log policy to log warnings and raise errors

compilation: Compilation = templater.compile(
    template_source=template_inline_source_settings,
    schema_source=schema_inline_source_settings
)

compilation.outcome

In [ ]:
compilation.message, compilation.diagnostics

### Rendering

Template

In [ ]:
from templisafe.util.util import ContentType

sql_template_content: str = """SELECT 
  {{ col1 }}, {{ col2 }}
FROM users u
WHERE TRUE
  AND u.status = '{{ user_status }}'
  AND u.age > {{ user_age_lower }}
"""

template_inline_source_settings: SourceSettings = SourceSettings.create(
    kind='inline',
    content=sql_template_content, 
    content_type=ContentType.TEXT
)
template_inline_source_settings

Schema

In [ ]:
from templisafe.util.util import ContentType

schema_content: str = """schema:
  col1: str
  col2: str
  user_status: str
  user_age_lower: int 
"""

schema_inline_source_settings: SourceSettings = SourceSettings.create(
    kind="inline",
    content=schema_content,
    content_type=ContentType.YAML
)

schema_inline_source_settings

In [ ]:
from templisafe.templater import Templater
from templisafe.templater_factory import TemplaterFactory

factory: TemplaterFactory = TemplaterFactory()
templater: Templater = factory.create(diagnostic_policy="ignore")

compilation: Compilation = templater.compile(
    template_source=template_inline_source_settings,
    schema_source=schema_inline_source_settings
)

compilation.outcome

In [ ]:
from templisafe.template.template_model import CompilationSpec

compiled: CompilationSpec = compilation.compiled
compiled

In [ ]:
compiled.schema.model_cls.model_fields

#### Undeclared parameters

In [ ]:
variants_content: str = """variants:
  col1: id
  col2: name
  user_status: ACTIVE
  user_age_lower: 18
  unused: unused 
"""

variants_inline_source_settings: SourceSettings = SourceSettings.create(
    kind='inline',
    content=variants_content, 
    content_type=ContentType.YAML
    )
variants_inline_source_settings

In [ ]:
from templisafe.template.template_model import Rendering

rendering: Rendering = templater.render(
    compiled=compiled,
    variants_sources=variants_inline_source_settings
)

rendering.outcome

In [ ]:
rendering.message, rendering.diagnostics

#### Wrong parameter type

In [ ]:
variants_content: str = """
variants:
  col1: 5.67                      # Should be a string
  col2: name
  user_status: 1                  # Should be a string
  user_age_lower: [1, 2, 3]       # Should be an int
"""

variants_inline_source_settings: SourceSettings = SourceSettings.create(
    kind='inline',
    content=variants_content, 
    content_type=ContentType.YAML
    )
variants_inline_source_settings

In [ ]:
from templisafe.template.template_model import Rendering

rendering: Rendering = templater.render(
    compiled=compiled,
    variants_sources=variants_inline_source_settings
)

rendering.outcome

In [ ]:
rendering.message, rendering.diagnostics